# Volume 8, Chapter 10: Integral Geometry (Python companion)

Bertrand's chords put to Jaynes's test, measuring the length of curves in images by counting crossings, and surface areas from shadows.

*Before each solution, predict what the output will look like.*

## Exercises

**P10.1 (★) Bertrand's chords and Jaynes's test.** (a) Simulate the three methods of Bertrand and the invariant measure (Corollary 10.2), and estimate the probability that the chord is longer than $\sqrt3$. (b) *Jaynes's test.* If the chords are produced by lines thrown without aiming, the lines that meet a smaller disc of radius $r$ inside the circle, centred anywhere, must cut it in chords distributed as the method itself predicts for that disc. Compare, for each method, the law of the rescaled distance $|p'|/r$ to the centre of the small disc with the law of $|p|$ for the unit disc, first for a concentric small disc and then for an off-centre one. Which method passes both tests?

**P10.2 (★★) CS application: measuring the length of a curve in an image.** A curve drawn in a binary image (a root, a neuron, a crack, a road on a map) has a length that pixel counting gets wrong: a diagonal segment of length $L$ touches about $\sqrt2L$ pixel widths of pixels, while a horizontal one touches $L$. (a) Rasterize curves of known length (a circle, a sine wave, a random spline) on a grid. Compare three estimators: the number of pixels, the length of the chain of pixel centres with steps $1$ and $\sqrt2$, and a *Crofton estimator* that counts crossings with the grid lines in four directions ($0^\circ$, $45^\circ$, $90^\circ$, $135^\circ$) and applies Theorem 10.4 with the $\theta$-integral replaced by an average over these directions. (b) How do the errors depend on the orientation of a straight segment? This is the stereological line-intersect method of Newman (1966), used to measure root lengths.

**P10.3 (★★) Shadows and surface area.** (a) Build random convex polytopes (convex hulls of random points on or in a sphere) with `scipy.spatial.ConvexHull`, compute their surface areas and the areas of their shadows in many random directions, and verify Theorem 10.9. (b) Estimate the surface area of a convex body from a few X-ray shadows only: how many random directions are needed for a $1\%$ error? (c) Check Barbier's theorem numerically on Reuleaux polygons with $3$, $5$ and $7$ vertices: their widths in all directions and their perimeters.

## Solutions

### P10.1

In [ ]:
import numpy as np
rng = np.random.default_rng(10)
M = 400000

def chords(method, n):
    """Return (theta, p) of n chords of the unit circle; |p| is the distance to the centre."""
    if method == "endpoints":
        a, b = rng.uniform(0, 2 * np.pi, n), rng.uniform(0, 2 * np.pi, n)
        return (a + b) / 2, np.cos((b - a) / 2)
    if method == "radius":
        return rng.uniform(0, np.pi, n), rng.uniform(-1, 1, n)
    if method == "midpoint":
        r, t = np.sqrt(rng.uniform(0, 1, n)), rng.uniform(0, 2 * np.pi, n)
        return t, r
    if method == "invariant lines":             # a big disc of lines around the circle, then keep those that hit it
        th, p = rng.uniform(0, np.pi, 4 * n), rng.uniform(-4, 4, 4 * n)
        cx, cy = rng.uniform(-1, 1, 2)          # the circle is drawn at an unknown place: shift does not matter
        p = p - (cx * np.cos(th) + cy * np.sin(th))
        keep = np.abs(p) <= 1
        return th[keep], p[keep]

for m in ("endpoints", "radius", "midpoint", "invariant lines"):
    th, p = chords(m, M)
    print(f"{m:16s}  P(chord > sqrt 3) = {np.mean(2 * np.sqrt(1 - p**2) > np.sqrt(3)):.4f}")

The three classical answers $\frac13$, $\frac12$, $\frac14$, and the invariant lines, thrown on the whole plane without knowing where the circle is, land on $\frac12$, with the same law as Bertrand's radius method.

In [ ]:
r = 0.4
bins = np.linspace(0, 1, 6)
print("law of the rescaled distance in five bins (unit disc, then small discs); max deviation from the unit-disc law")
for m in ("endpoints", "radius", "midpoint"):
    th, p = chords(m, M)
    big, _ = np.histogram(np.abs(p), bins, density=True)
    print(f"{m:10s} unit disc         {np.round(big, 2)}")
    for c in ((0.0, 0.0), (0.3, 0.0)):
        q = p - (c[0] * np.cos(th) + c[1] * np.sin(th))        # signed distance of the line to the small centre
        small, _ = np.histogram(np.abs(q[np.abs(q) < r]) / r, bins, density=True)
        print(f"{'':10s} centre {c}  {np.round(small, 2)}   deviation {np.max(np.abs(big - small)):.2f}")

The endpoint method fails already for the concentric disc: its law is not invariant under change of scale. The midpoint method passes the concentric test (a uniform point of the big disc that falls in the small concentric one is uniform there) but fails the off-centre one: it is not invariant under translations. Only the radius method passes both, because for each direction its lines have $p$ uniform, and so does their distance to any other point, which is $p$ shifted by a constant. This is Jaynes's argument; Theorem 10.1 says that rotations and translations alone already single out $\dd p\,\dd\theta$.

### P10.2

In [ ]:
from scipy.interpolate import CubicSpline

def rasterize(xs, ys, N):
    """Pixels of an N x N grid on [0,1]^2 visited by a densely sampled curve."""
    i = np.clip((xs * N).astype(int), 0, N - 1); j = np.clip((ys * N).astype(int), 0, N - 1)
    img = np.zeros((N, N), bool); img[i, j] = True
    return img

def curve_samples(kind, n=200000):
    t = np.linspace(0, 1, n)
    if kind == "circle":
        x, y = 0.5 + 0.35 * np.cos(2 * np.pi * t), 0.5 + 0.35 * np.sin(2 * np.pi * t)
    elif kind == "sine":
        x, y = 0.1 + 0.8 * t, 0.5 + 0.3 * np.sin(5 * np.pi * t)
    else:
        knots = np.random.default_rng(3).uniform(0.15, 0.85, (8, 2))
        cs = CubicSpline(np.linspace(0, 1, 8), knots)
        x, y = cs(t).T
    L = np.sum(np.hypot(np.diff(x), np.diff(y)))
    return x, y, L

def crofton_estimate(img):
    """Count transitions between empty and occupied pixels along rows, columns and both diagonals."""
    N = img.shape[0]; a = img.astype(int)
    counts = []
    counts.append(np.sum(np.abs(np.diff(a, axis=0))) / 2)   # each crossing of the curve gives two transitions
    counts.append(np.sum(np.abs(np.diff(a, axis=1))) / 2)
    d1 = sum(np.sum(np.abs(np.diff(np.diagonal(a, k)))) for k in range(-N + 1, N)) / 2
    d2 = sum(np.sum(np.abs(np.diff(np.diagonal(a[:, ::-1], k)))) for k in range(-N + 1, N)) / 2
    counts += [d1, d2]
    # lines in direction 0 and 90 degrees are spaced 1/N apart, the diagonal ones 1/(N sqrt 2)
    spacing = np.array([1, 1, 1 / np.sqrt(2), 1 / np.sqrt(2)]) / N
    # Crofton: E[crossings with a family of spacing s, random offset] = 2 L / (pi s) averaged over directions
    return np.pi / 2 * np.mean(np.array(counts) * spacing)

def chain_length(img):
    """Freeman chain approximation: link each pixel to its 8-neighbours, count each link once."""
    a = img.astype(int); N = a.shape[0]
    straight = np.sum(a[1:, :] & a[:-1, :]) + np.sum(a[:, 1:] & a[:, :-1])
    diag = np.sum(a[1:, 1:] & a[:-1, :-1]) + np.sum(a[1:, :-1] & a[:-1, 1:])
    return (straight + np.sqrt(2) * diag) / N

for kind in ("circle", "sine", "spline"):
    x, y, L = curve_samples(kind)
    img = rasterize(x, y, 400)
    print(f"{kind:7s} true length {L:.4f}:  pixel count {img.sum() / 400:.4f},  chain {chain_length(img):.4f},  Crofton {crofton_estimate(img):.4f}")

Counting the pixels a curve touches overestimates its length by about $25\%$ on these curves, since oblique stretches touch more pixels than their length; linking each pixel to its $8$ neighbours overestimates far more, because the drawn curve is a staircase with many redundant links. The Crofton estimator, which only counts how often the curve is crossed by the rows, columns and diagonals of the grid, is within $1\%$. A run of consecutive curve pixels along a row counts as one crossing, so the thickness of the digital curve does not matter. Its remaining error comes from averaging four directions instead of all of them, which is the next experiment.

In [ ]:
N = 400
print(" angle   pixel count / L   Crofton / L")
for deg in (0, 10, 22.5, 30, 45):
    a = np.radians(deg); t = np.linspace(0, 1, 200000)
    x, y = 0.2 + 0.6 * t * np.cos(a), 0.2013 + 0.6 * t * np.sin(a)     # generic position: not through pixel corners
    img = rasterize(x, y, N); L = 0.6
    print(f"{deg:6.1f}      {img.sum() / N / L:.3f}           {crofton_estimate(img) / L:.3f}")

For a straight segment the pixel count is exact only along the axes and grows to $\sqrt2\approx1.41$ times the length at $45^\circ$. The four-direction Crofton estimator stays between $0.95$ and $1.03$ for every angle. It replaces $\int_0^\pi|\cos(\theta-\varphi)|\dd\theta=2$ by the average $\frac\pi4\sum_k|\cos(\theta_k-\varphi)|$ over the four directions, whose ratio to $2$ is $0.948$ at $\varphi=0^\circ$ or $45^\circ$ and $1.026$ at $22.5^\circ$, exactly the ripple observed. More directions shrink it; this is how image-analysis software measures lengths.

### P10.3

In [ ]:
from scipy.spatial import ConvexHull

def shadow_area(points, u):
    u = u / np.linalg.norm(u)
    e1 = np.cross(u, [1, 0, 0] if abs(u[0]) < 0.9 else [0, 1, 0]); e1 /= np.linalg.norm(e1)
    e2 = np.cross(u, e1)
    return ConvexHull(np.c_[points @ e1, points @ e2]).volume     # the 'volume' of a planar hull is its area

for n in (8, 20, 200):
    pts = rng.standard_normal((n, 3)); pts /= np.linalg.norm(pts, axis=1, keepdims=True)
    hull = ConvexHull(pts); A = hull.area
    us = rng.standard_normal((4000, 3))
    mean_shadow = np.mean([shadow_area(pts[hull.vertices], u) for u in us])
    print(f"{n:4d} random points on the sphere: surface area {A:.4f}, A/4 = {A / 4:.4f}, mean shadow {mean_shadow:.4f}")

The mean shadow matches a quarter of the surface area, up to the Monte Carlo error of $4000$ directions. As the number of vertices grows the polytope approaches the sphere, with surface area $4\pi$ and every shadow equal to $\pi$.

In [ ]:
pts = rng.standard_normal((40, 3)) * [1.0, 0.6, 0.3]          # a flattened random body
hull = ConvexHull(pts); A = hull.area; V = pts[hull.vertices]
for k in (3, 10, 30, 100, 300):
    errs = []
    for _ in range(200):
        us = rng.standard_normal((k, 3))
        errs.append(4 * np.mean([shadow_area(V, u) for u in us]) / A - 1)
    print(f"{k:4d} shadows: typical relative error {np.std(errs) * 100:.1f}%")

The estimator $4\times(\text{mean shadow})$ is unbiased, and its error decreases like $k^{-1/2}$: for this rather flat body about a hundred random shadows give a few per cent, and $1\%$ needs on the order of a thousand. Choosing the directions on a well-spread design instead of at random (a quasi-Monte Carlo set on the sphere) does much better, as the smooth integrand $u\mapsto\operatorname{area}\pi_u(K)$ allows.

In [ ]:
def reuleaux_polygon(m, w=1.0, n=4000):
    """Boundary points of the Reuleaux polygon with m (odd) vertices and width w."""
    R = w / (2 * np.sin(np.pi * (m - 1) / (2 * m)))              # circumradius of the regular m-gon with diagonal w
    V = np.array([[R * np.cos(2 * np.pi * k / m), R * np.sin(2 * np.pi * k / m)] for k in range(m)])
    pts = []
    for k in range(m):
        c = V[k]; a1 = V[(k + (m - 1) // 2) % m]; a2 = V[(k + (m + 1) // 2) % m]
        t1, t2 = np.arctan2(*(a1 - c)[::-1]), np.arctan2(*(a2 - c)[::-1])
        if t2 < t1: t2 += 2 * np.pi
        tt = np.linspace(t1, t2, n // m)
        pts.append(np.c_[c[0] + w * np.cos(tt), c[1] + w * np.sin(tt)])
    return np.vstack(pts)

for m in (3, 5, 7):
    P = reuleaux_polygon(m)
    ths = np.linspace(0, np.pi, 721)
    widths = [np.ptp(P @ [np.cos(t), np.sin(t)]) for t in ths]
    hull = ConvexHull(P)
    print(f"m = {m}: width between {min(widths):.5f} and {max(widths):.5f};  perimeter {hull.area:.5f}  (pi = {np.pi:.5f})")

The arc of each Reuleaux polygon is centred at the opposite vertex with radius $w$, so every pair of parallel support lines is at distance $w$, and the perimeters all equal $\pi w$, as Barbier's theorem demands (the tiny defect is the polygonal approximation of the arcs). The Reuleaux triangle has the smallest area among sets of constant width $w$ (the Blaschke–Lebesgue theorem), the disc the largest; British $20$p and $50$p coins are Reuleaux heptagons, so that vending machines can measure them by their width in any orientation.